In [1]:
import kwant
import kwant.continuum
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import tinyarray
from scipy import constants

En este caso se tiene que el halmintoniano al cual se quiere simular es equivalente a: 

In [2]:
# Tenemos los siguintes parámetros físicos:
# Constantes físicas
hbar = constants.hbar  # J·s
eV = constants.eV      # J
nm = 1e-9              # m

# Parámetros del grafeno
a_graphene = 0.246     # nm - parámetro de red del grafeno
t_graphene = 2.7       # eV - hopping del grafeno prístino

# Parámetros del rGO (grafeno oxidado reducido)
# El rGO tiene defectos que modifican los parámetros de Dirac
v_F = 1e6              # m/s - velocidad de Fermi
D1 = 0.05              # eV·nm⁴ - término de cuarto orden (curvatura de banda)
D2 = -0.5              # eV·nm² - término de segundo orden (masa efectiva)
epsilon_rGO = 0.3      # eV - gap inducido por oxidación

# Funciones de trabajo (eV)
phi_Au = 5.1           # Oro - contacto débil
phi_Pd = 5.6           # Paladio - contacto fuerte
phi_rGO = 4.5          # rGO (aprox.)

# Parámetros de acoplamiento interfacial
t_interface_Au = 0.3   # eV - acoplamiento débil Au-rGO
t_interface_Pd = 1.2   # eV - acoplamiento fuerte Pd-rGO

In [3]:
# Se tiene las matrices de Pauli para el espacio sublattice
# Matrices de Pauli
sigma_0 = tinyarray.array([[1, 0], [0, 1]])
sigma_x = tinyarray.array([[0, 1], [1, 0]])
sigma_y = tinyarray.array([[0, -1j], [1j, 0]])
sigma_z = tinyarray.array([[1, 0], [0, -1]])

In [4]:
# El halmintoniano está dado por:
def get_rGO_hamiltonian_string():
    """
    Construye el Hamiltoniano continuo para rGO
    H = D₁∇⁴Ψ + D₂∇²Ψ + εΨ
    
    En términos de k: H = D₁k⁴ + D₂k² + ε + términos de Dirac
    """
    # Modelo de Dirac modificado con términos de orden superior
    ham_str = f"""
        {epsilon_rGO} * kron(sigma_0, sigma_0)
        + {D2} * (k_x**2 + k_y**2) * kron(sigma_0, sigma_0)
        + {D1} * (k_x**4 + k_y**4 + 2*k_x**2*k_y**2) * kron(sigma_0, sigma_0)
        + k_x * kron(sigma_z, sigma_x)
        + k_y * kron(sigma_z, sigma_y)
    """
    return ham_str